[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/induced_dipoles/induced_dipoles.ipynb)

# Induced Dipoles in a Cluster of Atoms

This example computes the electric dipoles induced in planar clusters of polarizable atoms by a uniform external electric field. Each atom polarizes in proportion to the total field at its location, combining the external field with the dipole fields radiated by every other atom in the cluster. Along a chain aligned with the external field, neighbouring dipole fields reinforce the applied field and amplify the polarization, whereas across a transverse chain they oppose the applied field and screen it; Silberstein introduced this mutual-polarization model in 1917 for diatomic molecules, and Purcell and Pennypacker extended it to oscillating optical fields in the discrete dipole approximation.

In spacetime, the electric field is modelled as an extrinsic bivector through the rest-frame time direction `time`, while each spatial separation and electric dipole moment is modelled as an intrinsic antivector orthogonal to `time`. Pairing a field with the worldsheet plane swept out by a dipole, `field & (dipole & time)`, reads the interaction energy directly, and dualizing `(Dipole & time).dual()` turns that worldsheet into the vacuum map `field_along` from dipole directions to electric fields. Sandwiching `Dipole` with a pairwise separation vector, `separation >> Dipole`, reflects a dipole across the plane normal to the separation so `field_along(squared_distance * Dipole - 3 * mirrors) / (2 * distance**5)` forms the pairwise dipole-field operator without coordinate outer products, and `response.solve(applied_field)` solves the coupled cluster at once.

In Gibbs–Heaviside vector and $3N \times 3N$ block-matrix notation, assembling the coupled cluster requires stacking $3 \times 3$ Cartesian outer-product blocks $\mathsf{T}_{ij} = \left(3\,\hat{\mathbf{r}}_{ij}\hat{\mathbf{r}}_{ij}^{\mathsf{T}} - \mathbf{I}_3\right) / r_{ij}^3$ for $i \neq j$ with $\mathsf{T}_{ii} = \mathbf{0}$ and solving $\left(\alpha^{-1}\mathbf{I}_{3N} - \mathsf{T}\right)\mathbf{p} = \mathbf{E}_0$ over flattened coordinate vectors $\mathbf{p}, \mathbf{E}_0 \in \mathbb{R}^{3N}$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.algebras import STA
from examples.util.animation import save_figure
from examples.electromagnetism.induced_dipoles import render

np.set_printoptions(precision=4, suppress=True)

ga = STA
mv = NumpyContext(ga).multivector
# Positions and dipoles (charge times displacement) are spatial directions: antivectors orthogonal to time.
Dipole = ga.gatype.vector().derive.within(Algebra("x+y+z+")).derive.dual        # Antivector-
Field = ga.gatype.bivector()                                                    # Bivector+
Rotor = ga.gatype.rotor()                                                       # Even+

# Rest-frame time direction of the cluster and vacuum map from a spatial direction to the electric field along it.
time = -mv.t.dual()                                                             # [] Antivector-
field_along = (Dipole & time).dual()                                            # [] Field+ <- Dipole-

## Planar atom clusters

Three planar arrangements place the atoms one unit of spacing apart in the `xy` plane: a longitudinal chain aligned with the applied field along `x`, a transverse chain turned a quarter turn across the field, and a regular ring with unit nearest-neighbour spacing. Stacking the three arrangements into one batch solves all three geometries together.

In [ ]:
# Units in which four pi times the vacuum permittivity equals one, with atomic polarizability well below self-polarization.
spacing = 1.0
polarizability = 0.1 * spacing**3
field_strength = 1.0
n_atoms = 7
case_names = ["chain along the field", "chain across the field", "ring"]
n_cases = len(case_names)

In [ ]:
offsets = spacing * (np.arange(n_atoms) - (n_atoms - 1) / 2)                    # [n_atoms]
longitudinal_chain = mv.x.dual() * offsets                                      # [n_atoms] Dipole-
quarter_turn = (mv.xy * (-np.pi / 4)).exp()                                     # [] Rotor+
transverse_chain = quarter_turn >> longitudinal_chain                           # [n_atoms] Dipole-
# Ring radius chosen so adjacent atoms around the polygon sit one spacing apart.
ring_radius = spacing / (2 * np.sin(np.pi / n_atoms))
ring_angles = 2 * np.pi * np.arange(n_atoms) / n_atoms                          # [n_atoms]
ring_turns = (mv.xy * (-ring_angles / 2)).exp()                                 # [n_atoms] Rotor+
ring = ring_turns >> (ring_radius * mv.x.dual())                                # [n_atoms] Dipole-
positions = stack([longitudinal_chain, transverse_chain, ring])                 # [n_cases, n_atoms] Dipole-

## Pairwise dipole coupling and cluster response

A static dipole produces an electric field that decays with the cube of distance, reaching twice the strength along the separation axis as perpendicular to it and reversing sign between the two directions. Sandwiching `Dipole` with the separation vector `separation >> Dipole` reflects any dipole in the plane perpendicular to the separation, reversing its longitudinal component while preserving its transverse component and scaling both by the squared distance. Subtracting three such reflections from `squared_distance * Dipole` and dividing by twice the fifth power of distance yields the pairwise coupling map, and subtracting that coupling from the single-atom inverse polarizability `field_along / polarizability` forms the full cluster response operator over all atom pairs.

In [ ]:
# Pairwise separations from each source atom to each target atom.
separation = positions[:, :, None] - positions[:, None, :]                      # [n_cases, n_atoms, n_atoms] Dipole-
squared_distance = separation.squared()                                         # [n_cases, n_atoms, n_atoms] Scalar+
# On the diagonal the separation vanishes along with the numerator, so adding an identity keeps the denominator non-zero.
distance = (squared_distance + np.eye(n_atoms)).square_root()                   # [n_cases, n_atoms, n_atoms] Scalar+
# Reflection in the plane normal to each separation reverses the longitudinal part and scales by squared distance.
mirrors = separation >> Dipole                                                  # [n_cases, n_atoms, n_atoms] Dipole- <- Dipole-
coupling = (field_along(squared_distance * Dipole - 3 * mirrors) / (2 * distance**5)).field(0, 1)  # [n_cases] (Field+ <- Dipole-)[n_atoms <- n_atoms]
# Single-atom inverse susceptibility on every site minus the inter-atomic dipole field coupling.
response = field_along / polarizability - coupling                              # [n_cases] (Field+ <- Dipole-)[n_atoms <- n_atoms]

print(response)

## Coupled dipoles and cluster polarization

Because every induced dipole radiates back onto its neighbours, the cluster solves as one coupled linear system against the uniform external field. Pairing the mean dipole's worldsheet plane with `applied_field` and dividing by the corresponding pairing for an isolated atom measures the enhancement or screening of each geometry.

In [ ]:
applied_field = field_along(field_strength * mv.x.dual())                       # [] Field+
dipoles = response.solve(applied_field)                                         # [n_cases] Dipole-[n_atoms]
isolated_dipole = polarizability * field_along.solve(applied_field)             # [] Dipole-
mean_dipole = dipoles.sites.mean()                                              # [n_cases] Dipole-
polarization_gain = (applied_field & (mean_dipole & time)) / (applied_field & (isolated_dipole & time))  # [n_cases] Scalar+

print("mean dipole per atom, relative to an isolated atom:")
print(case_names)
print(polarization_gain.to_array())

save_figure(render.clusters(positions, dipoles, isolated_dipole), "induced_dipoles");

Red arrows show the induced dipole on each dark atom dot, drawn to a common scale over the pale grey arrow of an isolated atom in the same external field. Along the longitudinal chain on the left, mutual reinforcement stretches the arrows past the isolated reference, peaking at the central atom where neighbours pull from both sides; across the transverse chain in the middle, depolarizing fields from neighbours shorten every arrow below the grey reference, strongest in the interior and weaker at the two open ends; around the ring on the right, atoms where the ring runs parallel to the field exceed the isolated reference, atoms where the ring runs across it fall short, and intermediate dipoles tilt toward the local tangent.

In [ ]:
# checks
# Each atom's dipole equals its polarizability times the local field (external plus neighbours' dipole fields).
local_fields = applied_field + coupling(dipoles)                                # [n_cases] Field+[n_atoms]
assert ((field_along(dipoles) / polarizability - local_fields).scalar_norm_squared() < 1e-20).all()
# Pairwise dipole interaction is reciprocal: reading response(p) against (q & time) matches reading response(q) against (p & time).
np.testing.assert_allclose((response.adjugate()(Dipole & time) + (time & response)).kernel, 0.0, atol=1e-12)
# Longitudinal alignment enhances polarization, transverse alignment screens it, and the planar ring lies strictly between.
longitudinal_gain, transverse_gain, ring_gain = polarization_gain
assert bool(longitudinal_gain > ring_gain > 1 > transverse_gain)
# Well below the polarization catastrophe, every normalized eigenvalue of the cluster response stays above one half.
assert ((polarizability * field_along.solve(response)).eigvalsh() > 1 / 2).all()